# Week 10B — Tracking Detected Objects

**MCTA 4364 Machine Vision** · Session 2 of Week 10 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W10_motion_tracking/W10B_tracking_detected_objects.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W10_motion_tracking/W10B_tracking_detected_objects.ipynb)

A detector (Weeks 8–9) answers *what is where* in **one** frame. To count parts on a conveyor, measure their speed, follow a person through a
store or hand a part to a robot, we need **identities over time**: *this* bolt in frame 12 is the same bolt in frame 13. This is
**multi-object tracking (MOT)** by the **tracking-by-detection** recipe used in practice: predict where each object will be (Kalman filter),
match predictions to new detections (Hungarian algorithm), and manage track births and deaths. You will build three trackers of increasing quality,
measure them against ground truth, and finish with YOLO + ByteTrack on a real video.

### Learning outcomes
By the end of this session you will be able to:
1. Explain tracking-by-detection and the roles of **prediction**, **association** and **track management**.
2. Implement a constant-velocity **Kalman filter** and explain how it bridges missed detections.
3. Solve the assignment problem with the **Hungarian algorithm** on an IoU cost matrix.
4. Build **SORT** and **ByteTrack**-style trackers and evaluate them with **MOTA** and **ID switches**.
5. Turn tracks into industrial measurements: **counts at a line** and **speed in mm/s**; run YOLO + ByteTrack on video.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–8 | 1–2 | Tracking-by-detection; a noisy detector on the conveyor |
| 8–18 | 3 | A naive tracker and its failures |
| 18–35 | 4–5 | Kalman filter (interactive); Hungarian assignment |
| 35–55 | 6–7 | SORT and ByteTrack; MOTA and ID switches (interactive) |
| 55–70 | 8–9 | Counting and speed; YOLO + ByteTrack on a street video |
| 70–80 | 12 | Quiz and exit ticket |
| Home | 10–11 | Exercises with self-checks, challenge |

**Hardware:** laptop CPU (about 2 minutes, mostly YOLO in Section 9). YOLO11n weights (6 MB) download once.

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly",
          "ultralytics": "ultralytics", "lap": "lap"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is missing. Install it with CUDA support as described in "
                      "SETUP.md, or use Google Colab where it is preinstalled.")
sys.path.append("resources/scripts")

In [ ]:
import time

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision
from scipy.optimize import linear_sum_assignment

import partsim
from cvhelpers import SMOKE, check, data_dir, interact, opencv_sample, pipeline_diagram, quiz, show

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
N_FRAMES = 60 if SMOKE else 200
BELT_SPEED = 6.0
frames, gt = partsim.make_conveyor_video(N_FRAMES, seed=2, speed=BELT_SPEED)
NAMES = partsim.SCENE_CLASSES
print(f"{N_FRAMES} frames, {len(set(np.concatenate([g['ids'] for g in gt])))} parts pass the camera")


def iou_matrix(a, b):
    a, b = np.asarray(a, np.float32).reshape(-1, 4), np.asarray(b, np.float32).reshape(-1, 4)
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    return torchvision.ops.box_iou(torch.tensor(a), torch.tensor(b)).numpy()

## 1. Tracking-by-detection

Every frame, a tracker keeps a list of **tracks** (an ID, a state such as position and velocity, and bookkeeping such as age and misses). Then:

1. **Predict** where every track should be now (motion model, e.g. constant velocity).
2. **Associate** the predictions with the new detections (a cost such as 1 − IoU, solved as an assignment problem).
3. **Update** matched tracks with their detection; count a **miss** for unmatched tracks.
4. **Manage**: start a new track for each unmatched detection (confirmed after a few hits); delete tracks that have missed too many frames.

The hard cases: **missed detections**, **false detections**, **occlusions**, objects that look alike, and **crossings**. A wrong association
produces an **ID switch**.

In [ ]:
fig, ax = plt.subplots(figsize=(22, 2.6))
pipeline_diagram(["frame t", "detector\n(boxes + scores)", "predict tracks\n(Kalman)", "associate\n(Hungarian, IoU)", "update matched\n+ count misses",
                  "birth / death\n(min hits, max age)", "tracks with IDs"],
                 colors=["#f1f3f4", "#e8f0fe", "#fef7e0", "#fce8e6", "#e6f4ea", "#e6f4ea", "#f1f3f4"], ax=ax, fontsize=10)
ax.set_title("Figure 1.1 - The tracking-by-detection loop (runs once per frame)", fontweight="bold")
plt.show()

## 2. A realistic, imperfect detector

To study tracking independently of the detector, we **simulate** one from the ground truth, with the typical errors of a real model (you can
swap in the Week 9B detector later): box jitter (σ = 2 px), 15% **missed** detections, a few **false positives**, and 30% of the true detections
with a **low confidence** (0.1–0.5), as happens for partly occluded or blurred objects.

In [ ]:
def simulated_detector(g, rng, p_miss=0.15, fp_rate=0.5, sigma=2.0, low_frac=0.3, size=(360, 640)):
    boxes, scores = [], []
    for b in g["boxes"]:
        if rng.random() < p_miss:
            continue
        boxes.append(b + rng.normal(0, sigma, 4))
        scores.append(rng.uniform(0.1, 0.5) if rng.random() < low_frac else rng.uniform(0.5, 0.95))
    for _ in range(rng.poisson(fp_rate)):
        x, y = rng.uniform(0, size[1] - 50), rng.uniform(0, size[0] - 50)
        w, h = rng.uniform(20, 50, 2)
        boxes.append([x, y, x + w, y + h]); scores.append(rng.uniform(0.1, 0.45))
    return np.array(boxes, np.float32).reshape(-1, 4), np.array(scores, np.float32)


def detections_for_video(seed=0, **kw):
    rng = np.random.default_rng(seed)
    return [simulated_detector(g, rng, **kw) for g in gt]


dets = detections_for_video(0)
fig, axes = plt.subplots(1, 3, figsize=(22, 4.6))
for ax, t in zip(axes, [N_FRAMES // 3, N_FRAMES // 3 + 1, N_FRAMES // 3 + 2]):
    vis = frames[t].copy()
    for b in gt[t]["boxes"]:
        cv2.rectangle(vis, tuple(b[:2].astype(int)), tuple(b[2:].astype(int)), (255, 255, 255), 1)
    for b, s in zip(*dets[t]):
        col = (0, 200, 0) if s >= 0.5 else (0, 165, 255)
        cv2.rectangle(vis, tuple(b[:2].astype(int)), tuple(b[2:].astype(int)), col, 2)
        cv2.putText(vis, f"{s:.2f}", (int(b[0]), int(b[1]) - 3), cv2.FONT_HERSHEY_SIMPLEX, 0.4, col, 1)
    ax.imshow(vis[..., ::-1]); ax.axis("off"); ax.set_title(f"frame {t}: {len(gt[t]['boxes'])} parts, {len(dets[t][0])} detections")
plt.suptitle("Figure 2.1 - White = ground truth, green = confident detection, orange = low-confidence detection (no IDs yet)", fontweight="bold")
plt.tight_layout(); plt.show()

## 3. A naive tracker: greedy IoU matching, no prediction

Match each track's **last box** to the detection with the highest IoU (if > 0.3); unmatched detections start new tracks; a track that misses a single
frame is deleted. Simple — and fragile: one missed detection kills the track, so the object gets a **new ID** when it is detected again.

In [ ]:
def greedy_tracker(dets, iou_thr=0.3, conf=0.5):
    tracks, next_id, out = [], 0, []
    for boxes, scores in dets:
        boxes = boxes[scores >= conf]
        M = iou_matrix([t["box"] for t in tracks], boxes)
        used = set()
        for i, t in enumerate(tracks):
            cand = [j for j in np.argsort(-M[i]) if j not in used] if len(boxes) else []
            if cand and M[i, cand[0]] > iou_thr:
                t["box"] = boxes[cand[0]]; t["miss"] = 0; used.add(cand[0])
            else:
                t["miss"] += 1
        tracks = [t for t in tracks if t["miss"] == 0]
        for j in range(len(boxes)):
            if j not in used:
                tracks.append(dict(id=next_id, box=boxes[j], miss=0)); next_id += 1
        out.append([(t["id"], t["box"].copy()) for t in tracks])
    return out


def draw_trajectories(out, ax, title, frame):
    centres = {}
    for o in out:
        for tid, b in o:
            centres.setdefault(tid, []).append(((b[0] + b[2]) / 2, (b[1] + b[3]) / 2))
    ax.imshow(frame[..., ::-1] // 2)
    cmap = plt.cm.tab20
    for tid, pts in centres.items():
        pts = np.array(pts)
        ax.plot(pts[:, 0], pts[:, 1], "-", color=cmap(tid % 20), lw=2)
        ax.plot(*pts[-1], "o", color=cmap(tid % 20), ms=4)
    ax.axis("off"); ax.set_title(f"{title}: {len(centres)} track IDs")


out_greedy = greedy_tracker(dets)
n_true = len(set(np.concatenate([g["ids"] for g in gt])))
fig, axes = plt.subplots(1, 2, figsize=(20, 5.5))
draw_trajectories([[(i, b) for i, b in zip(g["ids"], g["boxes"])] for g in gt], axes[0], "ground truth", frames[-1])
draw_trajectories(out_greedy, axes[1], "greedy IoU tracker", frames[-1])
plt.suptitle(f"Figure 3.1 - Every missed detection breaks a track: {n_true} true parts become many IDs (colour = ID)", fontweight="bold")
plt.tight_layout(); plt.show()

## 4. Prediction with a Kalman filter

The **Kalman filter** (Kalman, 1960) estimates a state $\mathbf{x}$ (here position and velocity) from noisy measurements, alternating two steps:

| step | equations | meaning |
|---|---|---|
| **predict** | $\hat{\mathbf{x}} = F\mathbf{x}$, $\;\hat P = F P F^\top + Q$ | move the state with the motion model; uncertainty grows by the process noise $Q$ |
| **update** | $K = \hat P H^\top (H \hat P H^\top + R)^{-1}$, $\;\mathbf{x} = \hat{\mathbf{x}} + K(\mathbf{z} - H\hat{\mathbf{x}})$, $\;P = (I - KH)\hat P$ | blend prediction and measurement $\mathbf{z}$, weighted by their uncertainties ($R$ = measurement noise) |

For a constant-velocity model in 1D, $\mathbf{x} = [p, v]^\top$, $F = \begin{bmatrix}1 & \Delta t\\0 & 1\end{bmatrix}$, $H = [1\;\;0]$.
When a detection is **missing**, we simply skip the update: the prediction carries the track through the gap, with growing uncertainty.

In [ ]:
def kalman_1d(zs, q=0.05, r=4.0, dt=1.0):
    x, P = np.array([zs[0] if not np.isnan(zs[0]) else 0.0, 0.0]), np.diag([r, 10.0])
    F, H, Q = np.array([[1, dt], [0, 1]]), np.array([[1.0, 0.0]]), q * np.array([[dt ** 3 / 3, dt ** 2 / 2], [dt ** 2 / 2, dt]])
    xs, sd = [], []
    for z in zs:
        x = F @ x; P = F @ P @ F.T + Q                                   # predict
        if not np.isnan(z):                                              # update only when there is a measurement
            K = P @ H.T / (H @ P @ H.T + r)
            x = x + (K * (z - H @ x)).ravel(); P = (np.eye(2) - K @ H) @ P
        xs.append(x.copy()); sd.append(np.sqrt(P[0, 0]))
    return np.array(xs), np.array(sd)


def kalman_demo(measurement_noise=2.0, gap_start=25, gap_length=10, q=0.05):
    t = np.arange(60)
    truth = 20 + BELT_SPEED * t
    rng = np.random.default_rng(0)
    z = truth + rng.normal(0, measurement_noise, len(t))
    z[gap_start:gap_start + gap_length] = np.nan                         # missed detections
    xs, sd = kalman_1d(z, q=q, r=measurement_noise ** 2)
    fig, axes = plt.subplots(1, 2, figsize=(17, 4.5))
    axes[0].plot(t, truth, "k--", label="true position"); axes[0].plot(t, z, "o", ms=3, label="measurements (gaps = missed)")
    axes[0].plot(t, xs[:, 0], lw=2, label="Kalman estimate")
    axes[0].fill_between(t, xs[:, 0] - 2 * sd, xs[:, 0] + 2 * sd, alpha=0.25, label="± 2 standard deviations")
    axes[0].set_xlabel("frame"); axes[0].set_ylabel("x (px)"); axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(t, xs[:, 1], lw=2, label="estimated velocity"); axes[1].axhline(BELT_SPEED, color="k", ls="--", label="true velocity")
    axes[1].axvspan(gap_start, gap_start + gap_length, color="#fce8e6", label="no detections"); axes[1].set_xlabel("frame")
    axes[1].set_ylabel("px / frame"); axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.suptitle("Figure 4.1 - A constant-velocity Kalman filter: smoothing, velocity estimation and bridging a gap", fontweight="bold")
    plt.tight_layout(); plt.show()


interact(kalman_demo, measurement_noise=widgets.FloatSlider(value=2.0, min=0.2, max=10, step=0.2), gap_start=widgets.IntSlider(value=25, min=5, max=45),
         gap_length=widgets.IntSlider(value=10, min=0, max=15), q=widgets.FloatLogSlider(value=0.05, base=10, min=-3, max=1))

For boxes, SORT uses the state $[c_x, c_y, w, h, v_x, v_y]$ (size assumed constant) and the same predict/update equations in 6D.

In [ ]:
class BoxKalman:
    """Constant-velocity Kalman filter on (cx, cy, w, h, vx, vy)."""
    def __init__(self, box):
        cx, cy, w, h = (box[0] + box[2]) / 2, (box[1] + box[3]) / 2, box[2] - box[0], box[3] - box[1]
        self.x = np.array([cx, cy, w, h, 0.0, 0.0])
        self.P = np.diag([10.0, 10.0, 10.0, 10.0, 100.0, 100.0])
        self.F = np.eye(6); self.F[0, 4] = self.F[1, 5] = 1.0
        self.H = np.eye(4, 6)
        self.Q = np.diag([1.0, 1.0, 1.0, 1.0, 0.5, 0.5])
        self.R = np.eye(4) * 4.0

    def predict(self):
        self.x = self.F @ self.x
        self.P = self.F @ self.P @ self.F.T + self.Q

    def update(self, box):
        z = np.array([(box[0] + box[2]) / 2, (box[1] + box[3]) / 2, box[2] - box[0], box[3] - box[1]])
        S = self.H @ self.P @ self.H.T + self.R
        K = self.P @ self.H.T @ np.linalg.inv(S)
        self.x = self.x + K @ (z - self.H @ self.x)
        self.P = (np.eye(6) - K @ self.H) @ self.P

    def box(self):
        cx, cy, w, h = self.x[:4]
        return np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])

## 5. Association: the Hungarian algorithm

Given $N$ predicted boxes and $M$ detections, build a cost matrix $C_{ij} = 1 - \mathrm{IoU}_{ij}$ and find the one-to-one assignment with the
**minimum total cost** (Kuhn, 1955; `scipy.optimize.linear_sum_assignment`, $O(n^3)$). Pairs whose IoU is below a threshold are rejected afterwards.
Greedy matching (best pair first) can be sub-optimal when objects are close together.

In [ ]:
M_toy = np.array([[0.90, 0.70, 0.00],
                  [0.80, 0.10, 0.00],
                  [0.00, 0.00, 0.60]])
g_pairs, used = [], set()
for i in range(3):                                             # greedy, row by row
    j = max((j for j in range(3) if j not in used), key=lambda j: M_toy[i, j]); g_pairs.append((i, j)); used.add(j)
r, c = linear_sum_assignment(-M_toy)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, pairs, name in [(axes[0], g_pairs, "greedy"), (axes[1], list(zip(r, c)), "Hungarian")]:
    ax.imshow(M_toy, cmap="Greens", vmin=0, vmax=1)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f"{M_toy[i, j]:.2f}", ha="center", va="center")
    for i, j in pairs:
        ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="#d93025", lw=3))
    ax.set_xlabel("detection"); ax.set_ylabel("track"); ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_title(f"{name}: total IoU {sum(M_toy[i, j] for i, j in pairs):.2f}")
plt.suptitle("Figure 5.1 - Greedy matching takes the best pair first and can lose overall", fontweight="bold")
plt.tight_layout(); plt.show()

## 6. SORT and ByteTrack

**SORT** (Bewley et al., 2016) = Kalman prediction + Hungarian IoU association + track management (`min_hits` before a track is reported,
`max_age` misses before it is deleted). It only uses **confident** detections.

**ByteTrack** (Zhang et al., 2022) adds one idea: **do not throw away low-confidence detections**. After matching the confident detections, it
tries to match the *remaining* tracks to the low-confidence ones. A partly occluded object often gets a low score but is still in the right place;
the track's prediction tells us which low-score boxes to trust.

**DeepSORT / BoT-SORT** add an **appearance embedding** (a re-identification CNN) to the cost, which helps after long occlusions and for people.

In [ ]:
def tracker(dets, mode="sort", conf=0.5, iou_thr=0.3, max_age=10, min_hits=2):
    """mode: 'sort' (confident detections only) or 'byte' (second association with low-confidence detections)."""
    tracks, next_id, out = [], 0, []

    def associate(trs, boxes):
        if not trs or len(boxes) == 0:
            return [], list(range(len(trs))), list(range(len(boxes)))
        M = iou_matrix([t["kf"].box() for t in trs], boxes)
        r, c = linear_sum_assignment(-M)
        pairs = [(i, j) for i, j in zip(r, c) if M[i, j] > iou_thr]
        mi, mj = {i for i, _ in pairs}, {j for _, j in pairs}
        return pairs, [i for i in range(len(trs)) if i not in mi], [j for j in range(len(boxes)) if j not in mj]

    for boxes, scores in dets:
        for t in tracks:
            t["kf"].predict()
        high, low = boxes[scores >= conf], boxes[scores < conf]
        pairs, un_t, un_d = associate(tracks, high)
        for i, j in pairs:
            tracks[i]["kf"].update(high[j]); tracks[i]["miss"] = 0; tracks[i]["hits"] += 1
        left = [tracks[i] for i in un_t]
        if mode == "byte":                                              # second association: remaining tracks x low-score boxes
            pairs2, un_t2, _ = associate(left, low)
            for i, j in pairs2:
                left[i]["kf"].update(low[j]); left[i]["miss"] = 0; left[i]["hits"] += 1
            left = [left[i] for i in un_t2]
        for t in left:
            t["miss"] += 1
        for j in un_d:                                                  # births from confident detections only
            tracks.append(dict(id=next_id, kf=BoxKalman(high[j]), miss=0, hits=1)); next_id += 1
        tracks = [t for t in tracks if t["miss"] <= max_age]
        out.append([(t["id"], t["kf"].box()) for t in tracks if t["miss"] == 0 and t["hits"] >= min_hits])
    return out


out_sort = tracker(dets, "sort")
out_byte = tracker(dets, "byte")
fig, axes = plt.subplots(1, 3, figsize=(24, 5))
draw_trajectories(out_greedy, axes[0], "greedy IoU", frames[-1])
draw_trajectories(out_sort, axes[1], "SORT (Kalman + Hungarian)", frames[-1])
draw_trajectories(out_byte, axes[2], "ByteTrack-style (+ low-score boxes)", frames[-1])
plt.suptitle(f"Figure 6.1 - Trajectories of the three trackers ({n_true} true parts)", fontweight="bold")
plt.tight_layout(); plt.show()

## 7. Evaluating trackers: MOTA and ID switches

Per frame, match tracker boxes to ground-truth objects (Hungarian, IoU ≥ 0.5) and count **false negatives** (missed objects), **false positives**
and **ID switches** (a ground-truth object matched to a different track ID than before). The CLEAR-MOT accuracy (Bernardin & Stiefelhagen, 2008):

$$\mathrm{MOTA} = 1 - \frac{\sum_t (\mathrm{FN}_t + \mathrm{FP}_t + \mathrm{IDSW}_t)}{\sum_t \mathrm{GT}_t}.$$

MOTA is dominated by detection errors; **IDF1** and **HOTA** weigh identity preservation more (see references).

In [ ]:
def clear_mot(out, gt_frames=gt):
    fn = fp = idsw = n_gt = 0
    last = {}
    for g, o in zip(gt_frames, out):
        n_gt += len(g["ids"])
        ob = np.array([b for _, b in o]).reshape(-1, 4); oid = [i for i, _ in o]
        M = iou_matrix(g["boxes"], ob)
        matches = []
        if M.size:
            r, c = linear_sum_assignment(-M)
            matches = [(i, j) for i, j in zip(r, c) if M[i, j] >= 0.5]
        fn += len(g["ids"]) - len(matches); fp += len(ob) - len(matches)
        for i, j in matches:
            gid = g["ids"][i]
            if gid in last and last[gid] != oid[j]:
                idsw += 1
            last[gid] = oid[j]
    return dict(MOTA=1 - (fn + fp + idsw) / max(n_gt, 1), FN=fn, FP=fp, IDSW=idsw, GT=n_gt)


results = {"greedy IoU": clear_mot(out_greedy), "SORT": clear_mot(out_sort), "ByteTrack-style": clear_mot(out_byte)}
fig, axes = plt.subplots(1, 2, figsize=(16, 4.3))
axes[0].bar(list(results), [r["MOTA"] for r in results.values()], color=["#9aa0a6", "#1a73e8", "#188038"]); axes[0].set_ylim(0, 1)
axes[0].set_title("MOTA (higher is better)"); axes[0].grid(alpha=0.3, axis="y")
xs = np.arange(3)
for k, (key, col) in enumerate([("FN", "#fbbc05"), ("FP", "#d93025"), ("IDSW", "#9334e6")]):
    axes[1].bar(xs + (k - 1) * 0.27, [r[key] for r in results.values()], 0.27, label=key, color=col)
axes[1].set_xticks(xs); axes[1].set_xticklabels(list(results)); axes[1].legend(); axes[1].set_title("error counts over the video"); axes[1].grid(alpha=0.3, axis="y")
plt.suptitle("Figure 7.1 - Prediction removes ID switches; using low-score detections recovers misses", fontweight="bold")
plt.tight_layout(); plt.show()
for k, r in results.items():
    print(f"{k:<16} MOTA {r['MOTA']:.3f} | FN {r['FN']:4d} | FP {r['FP']:3d} | ID switches {r['IDSW']:3d} | GT {r['GT']}")

### 🎛️ Interactive: how detector quality drives tracking quality

In [ ]:
def quality_demo(p_miss=0.15, low_frac=0.3, fp_rate=0.5, max_age=10):
    d = detections_for_video(1, p_miss=p_miss, low_frac=low_frac, fp_rate=fp_rate)
    r = {"greedy IoU": clear_mot(greedy_tracker(d)), "SORT": clear_mot(tracker(d, "sort", max_age=max_age)),
         "ByteTrack-style": clear_mot(tracker(d, "byte", max_age=max_age))}
    plt.figure(figsize=(8, 3.5))
    plt.bar(list(r), [v["MOTA"] for v in r.values()], color=["#9aa0a6", "#1a73e8", "#188038"])
    for k, (name, v) in enumerate(r.items()):
        plt.text(k, max(v["MOTA"], 0) + 0.02, f"MOTA {v['MOTA']:.2f}\nIDSW {v['IDSW']}", ha="center")
    plt.ylim(min(0, min(v["MOTA"] for v in r.values())), 1.15); plt.title(f"miss {p_miss:.0%}, low-score {low_frac:.0%}, FP rate {fp_rate}")
    plt.show()


interact(quality_demo, p_miss=widgets.FloatSlider(value=0.15, min=0, max=0.6, step=0.05), low_frac=widgets.FloatSlider(value=0.3, min=0, max=0.8, step=0.05),
         fp_rate=widgets.FloatSlider(value=0.5, min=0, max=5, step=0.5), max_age=widgets.IntSlider(value=10, min=0, max=30))

## 8. From tracks to measurements: counting and speed

Counting is the most common tracking application in industry (parts per minute, people entering a store, vehicles per lane). Count each **track
ID** once, when its centre **crosses a virtual line** in the right direction. Without tracking you would count the same part in every frame; with ID
switches you count it several times.

Each track also gives a **speed**: the Kalman velocity (px/frame) × mm/px × FPS.

In [ ]:
LINE_X = 320


def crossings(out, line_x=LINE_X):
    """IDs whose centre moves from left of the line to right of it."""
    last_x, counted = {}, set()
    for o in out:
        for tid, b in o:
            cx = (b[0] + b[2]) / 2
            if tid in last_x and last_x[tid] < line_x <= cx:
                counted.add(tid)
            last_x[tid] = cx
    return counted


true_count = len(crossings([[(i, b) for i, b in zip(g["ids"], g["boxes"])] for g in gt]))
counts = {k: len(crossings(o)) for k, o in [("greedy IoU", out_greedy), ("SORT", out_sort), ("ByteTrack-style", out_byte)]}
FPS, MM = 30.0, partsim.SCENE_MM_PER_PX
centres = {}
for o in out_byte:
    for tid, b in o:
        centres.setdefault(tid, []).append((b[0] + b[2]) / 2)
speeds = [np.median(np.diff(c)) * MM * FPS for c in centres.values() if len(c) > 10]
fig, axes = plt.subplots(1, 2, figsize=(17, 4.3))
axes[0].bar(["ground truth"] + list(counts), [true_count] + list(counts.values()), color=["k", "#9aa0a6", "#1a73e8", "#188038"])
axes[0].set_title(f"parts counted crossing x = {LINE_X}"); axes[0].grid(alpha=0.3, axis="y")
axes[1].hist(speeds, 20, color="#188038"); axes[1].axvline(BELT_SPEED * MM * FPS, color="k", ls="--", label=f"belt {BELT_SPEED * MM * FPS:.0f} mm/s")
axes[1].set_xlabel("track speed (mm/s)"); axes[1].legend(); axes[1].set_title("speed of every track (median frame-to-frame motion)")
plt.suptitle("Figure 8.1 - Counting at a line and measuring speed from tracks", fontweight="bold")
plt.tight_layout(); plt.show()
print(f"true count {true_count} | " + " | ".join(f"{k}: {v}" for k, v in counts.items()))

## 9. Real video: YOLO + ByteTrack

Ultralytics wraps detection and tracking in one call: `model.track(frame, persist=True, tracker="bytetrack.yaml")` (or `"botsort.yaml"`, which adds
camera-motion compensation and optional re-identification). `persist=True` keeps the tracker state between calls. We count people crossing a
horizontal line in `vtest.avi`.

In [ ]:
from ultralytics import YOLO

yolo = YOLO(str(data_dir("weights") / "yolo11n.pt"))
cap = cv2.VideoCapture(str(opencv_sample("vtest.avi")))
LINE_Y = 330
history, crossed, frames_shown = {}, set(), []
n_video = 40 if SMOKE else 250
t0 = time.perf_counter()
for i in range(n_video):
    ok, frame = cap.read()
    if not ok:
        break
    r = yolo.track(frame, persist=True, tracker="bytetrack.yaml", classes=[0], conf=0.3, verbose=False)[0]
    if r.boxes.id is not None:
        for tid, b in zip(r.boxes.id.int().tolist(), r.boxes.xyxy.cpu().numpy()):
            c = ((b[0] + b[2]) / 2, b[3])                                 # bottom centre = feet
            if tid in history and history[tid][-1][1] < LINE_Y <= c[1]:
                crossed.add(tid)
            history.setdefault(tid, []).append(c)
    if i in (n_video // 3, 2 * n_video // 3, n_video - 1):
        vis = r.plot()
        cv2.line(vis, (0, LINE_Y), (vis.shape[1], LINE_Y), (0, 255, 255), 2)
        for tid, pts in history.items():
            if len(pts) > 1:
                cv2.polylines(vis, [np.array(pts, np.int32)], False, (255, 0, 255), 2)
        frames_shown.append((i, vis))
cap.release()
dt = (time.perf_counter() - t0) / max(i + 1, 1)
fig, axes = plt.subplots(1, len(frames_shown), figsize=(22, 5.5))
for ax, (i, vis) in zip(np.atleast_1d(axes), frames_shown):
    ax.imshow(vis[..., ::-1]); ax.axis("off"); ax.set_title(f"frame {i}")
plt.suptitle(f"Figure 9.1 - YOLO11n + ByteTrack: {len(history)} person IDs, {len(crossed)} crossed the yellow line downwards "
             f"({1 / dt:.1f} FPS on this machine)", fontweight="bold")
plt.tight_layout(); plt.show()

## 10. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Hungarian association
Given predicted boxes and detections (xyxy), return a list of `(track_index, detection_index)` pairs from the optimal assignment on the IoU matrix, keeping only pairs with IoU **>** `iou_thr`. Use `iou_matrix` and `linear_sum_assignment`.

In [ ]:
def associate(pred_boxes, det_boxes, iou_thr=0.3):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    preds = np.array([[0, 0, 10, 10], [20, 0, 30, 10], [100, 100, 110, 110]], float)
    detsb = np.array([[21, 0, 31, 10], [1, 0, 11, 10], [50, 50, 60, 60]], float)
    got = sorted((int(i), int(j)) for i, j in associate(preds, detsb))
    assert got == [(0, 1), (1, 0)], f"expected [(0, 1), (1, 0)], got {got}"
    assert list(associate(preds, np.zeros((0, 4)))) == [], "no detections -> no pairs"
    toy_pred = np.array([[0, 0, 10, 10], [0, 0, 10, 9]], float); toy_det = np.array([[0, 0, 10, 9.5], [0, 0, 10, 8]], float)
    M = iou_matrix(toy_pred, toy_det); r, c = linear_sum_assignment(-M)
    assert sorted((int(i), int(j)) for i, j in associate(toy_pred, toy_det)) == sorted(zip(r.tolist(), c.tolist())), "must be the optimal assignment"


check("E1 associate", _test_e1)

### E2. One Kalman step (1D constant velocity)
Implement `kalman_step(x, P, z, dt=1.0, q=0.05, r=4.0)` for state `x = [p, v]` with the matrices of Section 4 (process noise $Q = q\begin{bmatrix}dt^3/3 & dt^2/2\\ dt^2/2 & dt\end{bmatrix}$). Predict, then update with measurement `z` unless `z` is `None`. Return `(x, P)`.

In [ ]:
def kalman_step(x, P, z, dt=1.0, q=0.05, r=4.0):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    x, P = np.array([0.0, 6.0]), np.diag([4.0, 1.0])
    x1, P1 = kalman_step(x, P, None)
    assert np.allclose(x1, [6, 6]) and P1[0, 0] > P[0, 0], "without a measurement: move by v*dt and grow the uncertainty"
    x2, P2 = kalman_step(x, P, 7.0)
    assert 6 < x2[0] < 7 and P2[0, 0] < P1[0, 0], "with a measurement: the estimate moves towards z and uncertainty shrinks"
    zs = np.r_[20 + 6 * np.arange(30.0)]
    xs, _ = kalman_1d(zs)
    xk, Pk = np.array([zs[0], 0.0]), np.diag([4.0, 10.0])
    for z in zs:
        xk, Pk = kalman_step(xk, Pk, z)
    assert np.allclose(xk, xs[-1], atol=1e-6), "differs from kalman_1d in Section 4"


check("E2 kalman_step", _test_e2)

### E3. Line-crossing counter
Given `tracks` = a dict `{track_id: [(x, y), (x, y), ...]}` of centre positions in time order, return the **number of IDs** that cross `line_x` from left to right (a step with `x_prev < line_x <= x_next`). Count each ID at most once.

In [ ]:
def count_crossings(tracks, line_x):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    tr = {1: [(0, 5), (5, 5), (12, 5)], 2: [(15, 0), (8, 0)], 3: [(9, 0), (10, 0), (9, 0), (11, 0)], 4: [(10, 0), (11, 0)]}
    assert count_crossings(tr, 10) == 2, f"IDs 1 and 3 cross (3 only once), got {count_crossings(tr, 10)}"
    centres = {}
    for o in out_byte:
        for tid, b in o:
            centres.setdefault(tid, []).append(((b[0] + b[2]) / 2, (b[1] + b[3]) / 2))
    assert count_crossings(centres, LINE_X) == len(crossings(out_byte)), "differs from Section 8 on the conveyor"


check("E3 count_crossings", _test_e3)

### E4. Plug in a real detector (no self-check)
Replace `simulated_detector` by the Week 9B detector (`YOLO('data/w09b/runs/conveyor/weights/best.pt')`) run on `frames`, with `conf=0.1` so that
low-score boxes are kept. Compare MOTA for SORT and ByteTrack-style tracking, and the counts at the line. Which errors dominate now?

In [ ]:
# E4: your code here

## 11. Challenge: occlusion and re-identification
Make two parts overlap for 10 frames (spawn them at the same x on nearby rows with different speeds, e.g. by editing a copy of
`make_conveyor_video`). Measure the ID switches of SORT. Then add an **appearance** term to the cost: a colour or HOG descriptor of each track's last
crop, combined as `cost = 0.7 (1 - IoU) + 0.3 (1 - cosine similarity)`. Do the switches disappear? (This is the idea of DeepSORT.)

In [ ]:
# Your challenge code here

## 12. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'What are the per-frame steps of tracking-by-detection?',
        "options": [
            'Detect, predict, associate, update and manage tracks',
            'Detect and draw boxes',
            'Segment and classify',
            'Calibrate and undistort',
        ],
        "answer": 'dd7efe2b93',
        "explain": 'Prediction and association give identities over time.',
    },
    {
        "q": 'What does the Kalman filter do when a detection is missing?',
        "options": [
            'Deletes the track immediately',
            'Predicts the state forward and increases its uncertainty',
            'Uses the previous box unchanged',
            'Stops the video',
        ],
        "answer": '6f02fae57a',
        "explain": 'The update step is skipped; the prediction bridges the gap.',
    },
    {
        "q": 'What does the Kalman gain K balance?',
        "options": [
            'Speed and accuracy',
            'The uncertainty of the prediction and of the measurement',
            'IoU and confidence',
            'Frame rate and resolution',
        ],
        "answer": 'e2c7358609',
        "explain": 'Trust the measurement more when the prediction is uncertain, and vice versa.',
    },
    {
        "q": 'Which algorithm finds the optimal one-to-one matching between tracks and detections?',
        "options": [
            'NMS',
            'Hungarian (Kuhn-Munkres)',
            'RANSAC',
            'k-means',
        ],
        "answer": 'f5a0b09464',
        "explain": 'scipy.optimize.linear_sum_assignment.',
    },
    {
        "q": 'What is the key idea of ByteTrack?',
        "options": [
            'Use a transformer',
            'Also associate low-confidence detections with unmatched tracks',
            'Track in 3D',
            'Use optical flow',
        ],
        "answer": 'd3071269ec',
        "explain": 'Occluded objects often get low scores but are in the predicted place.',
    },
    {
        "q": 'An object keeps its box but changes track ID between frames 10 and 11. MOT calls this...',
        "options": [
            'a false positive',
            'an ID switch',
            'a false negative',
            'a merge',
        ],
        "answer": '9dea16f991',
        "explain": 'It hurts identity-based metrics (IDF1, HOTA) and counts.',
    },
    {
        "q": 'Why must counting use track IDs rather than per-frame detections?',
        "options": [
            'It is faster',
            'Otherwise the same object is counted in every frame',
            'Detections have no class',
            'IDs improve mAP',
        ],
        "answer": '754c78dcf2',
        "explain": 'Count each ID once when it crosses the line.',
    },
], title='Week 10B quiz')

## 13. Exit ticket (reflection)
1. Your line counter over-counts by 20%. Name two likely causes and how you would confirm each.
2. When would you prefer BoT-SORT (with re-identification) over ByteTrack?
3. The belt stops for 10 seconds. What happens to a constant-velocity Kalman tracker, and how would you fix it?

## 14. What's new (2025–26)

> - **Motion-aware and appearance-light trackers** (OC-SORT, Deep OC-SORT, Hybrid-SORT, BoostTrack) keep the SORT recipe but fix its weaknesses
>   under occlusion and non-linear motion; ByteTrack and BoT-SORT remain the defaults in Ultralytics.
> - **End-to-end transformer trackers** (MOTR, MOTRv2, MOTIP) and **SAM 2**-based trackers (SAMURAI, SAM2MOT) track masks of arbitrary objects
>   from a single prompt.
> - **Any-point trackers** (CoTracker 3, TAPNext) follow individual pixels through long occlusions, useful for deformable objects and robotics.
> - Evaluation moved from MOTA to **HOTA** (balances detection and association) and **IDF1**; TrackEval is the reference implementation.

## 15. References and further exploration
- Kalman, R. E. (1960). *A new approach to linear filtering and prediction problems.* J. Basic Engineering 82(1). [doi:10.1115/1.3662552](https://doi.org/10.1115/1.3662552)
- Welch, G. & Bishop, G. (2006). *An introduction to the Kalman filter.* UNC TR 95-041. [cs.unc.edu/~welch/kalman](https://www.cs.unc.edu/~welch/kalman/)
- Kuhn, H. W. (1955). *The Hungarian method for the assignment problem.* Naval Research Logistics 2. [doi:10.1002/nav.3800020109](https://doi.org/10.1002/nav.3800020109)
- Bewley, A. et al. (2016). *Simple online and realtime tracking* (SORT). ICIP. [arXiv:1602.00763](https://arxiv.org/abs/1602.00763)
- Wojke, N., Bewley, A. & Paulus, D. (2017). *Simple online and realtime tracking with a deep association metric* (DeepSORT). [arXiv:1703.07402](https://arxiv.org/abs/1703.07402)
- Zhang, Y. et al. (2022). *ByteTrack: Multi-object tracking by associating every detection box.* ECCV. [arXiv:2110.06864](https://arxiv.org/abs/2110.06864)
- Aharon, N., Orfaig, R. & Bobrovsky, B.-Z. (2022). *BoT-SORT: Robust associations multi-pedestrian tracking.* [arXiv:2206.14651](https://arxiv.org/abs/2206.14651)
- Cao, J. et al. (2023). *Observation-centric SORT* (OC-SORT). CVPR. [arXiv:2203.14360](https://arxiv.org/abs/2203.14360)
- Bernardin, K. & Stiefelhagen, R. (2008). *Evaluating multiple object tracking performance: the CLEAR MOT metrics.* EURASIP JIVP. [doi:10.1155/2008/246309](https://doi.org/10.1155/2008/246309)
- Ristani, E. et al. (2016). *Performance measures and a data set for multi-target, multi-camera tracking* (IDF1). [arXiv:1609.01775](https://arxiv.org/abs/1609.01775)
- Luiten, J. et al. (2021). *HOTA: A higher order metric for evaluating multi-object tracking.* IJCV 129. [arXiv:2009.07736](https://arxiv.org/abs/2009.07736)
- Ultralytics *Track mode* documentation (ByteTrack, BoT-SORT configuration). [docs.ultralytics.com/modes/track](https://docs.ultralytics.com/modes/track/)

**Data:** `vtest.avi` from the OpenCV sample data; conveyor video generated by `partsim.py`.

## 16. Key takeaways
- Tracking-by-detection = **predict** (Kalman) + **associate** (Hungarian on IoU) + **update** + **manage** births and deaths.
- The Kalman filter smooths boxes, estimates **velocity** and **bridges missed detections**.
- **SORT** removes most ID switches; **ByteTrack** also recovers low-confidence detections; appearance features (DeepSORT, BoT-SORT) help after long occlusions.
- Evaluate with **MOTA**, ID switches (and IDF1/HOTA); tracking turns detections into **counts** and **speeds**.

**Next week (11A):** from boxes to pixels — deep semantic and instance segmentation.